<a href="https://colab.research.google.com/github/KieranB2004/MECE694proj/blob/main/GroupPrject_code_MECE694.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd, glob
from pathlib import Path

# uncomment depending who is running the code
# D = Path("/content/drive/MyDrive/IAQ Research/BAVIAQ_Residential_Homes/Raw_Data/Extracted/IAQ_Activity_Monitoring/IAQ_Activity_Monitoring")
D = Path("")

# 1) how many files, and is there a ReadMe?
files = sorted(glob.glob(str(D / "*")))
print(len(files), "files")
print([Path(f).name for f in files if not Path(f).name.startswith("DataTable")])

# 2) column names in one home
df = pd.read_csv(D / "DataTable401.csv", nrows=5, low_memory=False)
print(len(df.columns), "columns")
for c in df.columns:
    print(c)
print(df.head(3).T.head(15))

0 files
[]


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IAQ Research/BAVIAQ_Residential_Homes/Raw_Data/Extracted/IAQ_Activity_Monitoring/IAQ_Activity_Monitoring/DataTable401.csv'

In [ ]:
"""
D7 (BAVIAQ) cooking-label check  -  MECE 694 project, Week 1, step 5
Colab version: paste this whole file into one cell and run it.

What it does
  1. Prints the README lines about the cooking sensors (IBU_ / PTR_ columns).
  2. Loads every DataTable*.csv (one file per home-week, 1-min data).
  3. Turns cooktop/oven temperatures into cooking ON/OFF:
        ON when a sensor is more than DELTA_C above its own recent baseline.
  4. Groups ON minutes into cooking events and checks whether PM2.5 rises.
  5. Prints a summary per home-week, saves CSVs and one example plot.

Labels: IBU_* = iButton temperature on burners/oven, PTR_* = power logger on
other cooking devices (not in every home). ANM_RHD = range hood (used only
as a cross-check, not as a label).
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================== CONFIG ==============================
BASE = Path("/content/drive/MyDrive/IAQ Research/BAVIAQ_Residential_Homes")
DATA_DIR = BASE / "Raw_Data/Extracted/IAQ_Activity_Monitoring/IAQ_Activity_Monitoring"
OUT_DIR = BASE / "Results/MECE694_label_check"

TIME_COL = "Time"
TIME_FORMAT = "%m/%d/%Y %H:%M"           # e.g. 9/17/2018 18:00
PM_COLS = ["PMin", "AVP_IN1_PM25"]       # first one that exists and has data is used

DELTA_C = 8.0         # IBU sensor is "cooking" when this many deg C above its baseline
BASELINE_MIN = 180    # baseline = 10th percentile of the previous 180 min
PTR_ON_W = 20.0       # PTR power logger is "cooking" above this many watts
MIN_EVENT_MIN = 3     # ignore ON periods shorter than this
MIN_GAP_MIN = 10      # merge events closer than this
PM_BASE_MIN = 30      # PM2.5 baseline = median of 30 min before the event
PM_AFTER_MIN = 30     # look for the PM2.5 peak until 30 min after the event
RISE_UGM3 = 5.0       # event "shows in PM2.5" if peak - baseline >= this
PLOT_FILE = None      # e.g. "DataTable401" (None = file with the most events)
PLOT_DAYS = 3
# ====================================================================


def show_readme():
    """Print the README lines that explain the cooking sensor columns."""
    for f in DATA_DIR.glob("*readme*.txt"):
        print(f"--- {f.name}: lines about IBU / PTR / cooking ---")
        lines = f.read_text(errors="ignore").splitlines()
        for i, line in enumerate(lines):
            if re.search(r"IBU|PTR|cook|oven|burner|RHD", line, re.I):
                print(f"{i:4d}: {line.strip()}")
        print()


def load_file(path):
    df = pd.read_csv(path, low_memory=False)
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
    df["time"] = pd.to_datetime(df[TIME_COL], format=TIME_FORMAT, errors="coerce")
    df = df.dropna(subset=["time"]).sort_values("time").set_index("time")
    for c in df.columns:
        if c != TIME_COL:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def cooking_on(df):
    """Return (on: bool Series, per-sensor ON Series dict)."""
    per = {}
    for c in [c for c in df.columns if c.startswith("IBU_")]:
        t = df[c]
        if t.notna().sum() == 0:
            continue
        base = t.rolling(f"{BASELINE_MIN}min", min_periods=10).quantile(0.10).shift(1)
        per[c] = (t - base) > DELTA_C
    for c in [c for c in df.columns if c.startswith("PTR_")]:
        if df[c].notna().sum():
            per[c] = df[c] > PTR_ON_W
    if not per:
        return pd.Series(False, index=df.index), per
    on = pd.concat(per, axis=1).fillna(False).any(axis=1)
    return on, per


def to_events(on, file_id):
    """Group consecutive ON minutes into events; drop short ones; merge close ones."""
    on = on.astype(bool)
    starts = on.index[on & ~on.shift(fill_value=False)]
    ends = on.index[on & ~on.shift(-1, fill_value=False)]
    ev = []
    for s, e in zip(starts, ends):
        if ev and s - ev[-1][1] <= pd.Timedelta(minutes=MIN_GAP_MIN):
            ev[-1][1] = e
        else:
            ev.append([s, e])
    ev = pd.DataFrame(ev, columns=["start", "end"])
    ev["start"] = pd.to_datetime(ev["start"])
    ev["end"] = pd.to_datetime(ev["end"])
    ev["duration_min"] = (ev.end - ev.start).dt.total_seconds() / 60 + 1
    ev = ev[ev.duration_min >= MIN_EVENT_MIN].reset_index(drop=True)
    ev.insert(0, "file", file_id)
    return ev


def pm_response(pm, ev):
    rows = []
    for _, r in ev.iterrows():
        b = pm[r.start - pd.Timedelta(minutes=PM_BASE_MIN): r.start].median()
        w = pm[r.start: r.end + pd.Timedelta(minutes=PM_AFTER_MIN)].dropna()
        peak = w.max() if len(w) else np.nan
        delay = (w.idxmax() - r.start).total_seconds() / 60 if len(w) else np.nan
        rows.append((b, peak, delay))
    ev = ev.copy()
    rows = pd.DataFrame(rows, index=ev.index, columns=["pm_baseline", "pm_peak", "peak_delay_min"])
    for c in rows.columns:
        ev[c] = rows[c].astype(float)
    ev["pm_rise"] = ev.pm_peak - ev.pm_baseline
    ev["shows_in_pm"] = ev.pm_rise >= RISE_UGM3
    return ev


def pick_pm(df):
    for c in PM_COLS:
        if c in df.columns and df[c].notna().mean() > 0.5:
            return c
    return None


def plot_file(df, ev, file_id, pm_col):
    t0 = ev.start.min().normalize() if len(ev) else df.index.min()
    t1 = t0 + pd.Timedelta(days=PLOT_DAYS)
    d = df[t0:t1]
    e = ev[(ev.start >= t0) & (ev.start < t1)]
    ibu = [c for c in d.columns if c.startswith("IBU_") and d[c].notna().any()]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    ax1.plot(d.index, d[pm_col], lw=0.8, color="#1f4e79")
    ax1.set_ylabel(f"{pm_col} (µg/m³)")
    for c in ibu:
        ax2.plot(d.index, d[c], lw=0.8, label=c)
    ax2.set_ylabel("Cooktop / oven temp (°C)")
    ax2.legend(loc="upper right", ncol=3, fontsize=8)
    for ax in (ax1, ax2):
        for _, r in e.iterrows():
            ax.axvspan(r.start, r.end, color="#e07b39", alpha=0.25)
    ax1.set_title(f"{file_id}: PM2.5 (top) and cooking sensors (bottom); "
                  f"orange = detected cooking")
    fig.tight_layout()
    path = OUT_DIR / f"{file_id}_pm25_cooking.png"
    fig.savefig(path, dpi=150)
    plt.show()
    print(f"Saved plot: {path}")


def run():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    show_readme()
    files = sorted(DATA_DIR.glob("DataTable*.csv"))
    print(f"{len(files)} home-week files\n")

    all_ev, summary, cache = [], [], {}
    for f in files:
        file_id = f.stem
        home = re.search(r"\d+", file_id).group()
        df = load_file(f)
        pm_col = pick_pm(df)
        on, per = cooking_on(df)
        ev = to_events(on, file_id)
        if pm_col:
            ev = pm_response(df[pm_col], ev)
        ev.insert(1, "home", home)
        all_ev.append(ev)
        days = (df.index.max() - df.index.min()).total_seconds() / 86400
        summary.append({
            "file": file_id, "home": home, "days": round(days, 1),
            "cook_sensors": ",".join(per.keys()) or "NONE",
            "pm_col": pm_col or "NONE",
            "n_events": len(ev),
            "events_per_day": round(len(ev) / days, 1) if days else np.nan,
            "pct_minutes_cooking": round(100 * on.mean(), 2),
            "pct_events_pm_rise": round(100 * ev.shows_in_pm.mean(), 0) if len(ev) and pm_col else np.nan,
            "median_pm_rise": round(ev.pm_rise.median(), 1) if len(ev) and pm_col else np.nan,
            "median_peak_delay_min": round(ev.peak_delay_min.median(), 0) if len(ev) and pm_col else np.nan,
        })
        cache[file_id] = (df, ev, pm_col)

    ev = pd.concat(all_ev, ignore_index=True)
    s = pd.DataFrame(summary)
    ev.to_csv(OUT_DIR / "cooking_events.csv", index=False)
    s.to_csv(OUT_DIR / "summary_per_file.csv", index=False)

    pd.set_option("display.width", 200)
    print(s.to_string(index=False))

    print("\n================ SUMMARY ================")
    print(f"Home-week files:                  {len(s)}  ({s.home.nunique()} homes)")
    print(f"Files with no cooking sensors:    {(s.cook_sensors == 'NONE').sum()}")
    print(f"Files with >= 10 cooking events:  {(s.n_events >= 10).sum()}")
    print(f"Total cooking events:             {len(ev)}")
    print(f"Median events per day:           {s.events_per_day.median()}")
    print(f"Cooking share of minutes:         {s.pct_minutes_cooking.median()}% (median per file)")
    if "shows_in_pm" in ev:
        print(f"Events with PM2.5 rise >= {RISE_UGM3}:   {100 * ev.shows_in_pm.mean():.0f}%")
        print(f"Median delay to PM2.5 peak:       {ev.peak_delay_min.median():.0f} min")
    print("\nGo/no-go guide:")
    print("  GO    most files have >= 10 events and 1-4 events/day, and many events show a PM2.5 rise")
    print("  TUNE  far too many events/day -> raise DELTA_C; almost none -> lower DELTA_C")
    print("  NO-GO most files have no cooking sensors or no events -> fallback (occupancy main task)")

    pick = PLOT_FILE or s.sort_values("n_events", ascending=False).file.iloc[0]
    df, e, pm_col = cache[pick]
    if pm_col:
        plot_file(df, e, pick, pm_col)
    return s, ev


summary, events = run()

In [ ]:
"""
D7 (BAVIAQ) cooking-label check  -  MECE 694 project, Week 1, step 5
Colab version: paste this whole file into one cell and run it.

What it does
  1. Prints the README lines about the cooking sensors (IBU_ / PTR_ columns).
  2. Loads every DataTable*.csv (one file per home-week, 1-min data).
  3. Turns cooktop/oven temperatures into cooking ON/OFF:
        ON when a sensor is more than DELTA_C above its own recent baseline.
  4. Groups ON minutes into cooking events and checks whether PM2.5 rises.
  5. Prints a summary per home-week, saves CSVs and one example plot.

Labels: IBU_* = iButton temperature on burners/oven, PTR_* = power logger on
other cooking devices (not in every home). ANM_RHD = range hood (used only
as a cross-check, not as a label).
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================== CONFIG ==============================
BASE = Path("/content/drive/MyDrive/IAQ Research/BAVIAQ_Residential_Homes")
DATA_DIR = BASE / "Raw_Data/Extracted/IAQ_Activity_Monitoring/IAQ_Activity_Monitoring"
OUT_DIR = BASE / "Results/MECE694_label_check"

TIME_COL = "Time"
TIME_FORMAT = "%m/%d/%Y %H:%M"           # e.g. 9/17/2018 18:00
PM_COLS = ["PMin", "AVP_IN1_PM25"]       # first one that exists and has data is used

DELTA_C = 8.0         # IBU sensor is "cooking" when this many deg C above its baseline
BASELINE_MIN = 180    # baseline = 10th percentile of the previous 180 min
PTR_ON_W = 20.0       # PTR power logger is "cooking" above this many watts
MIN_EVENT_MIN = 3     # ignore ON periods shorter than this
MIN_GAP_MIN = 10      # merge events closer than this
PM_BASE_MIN = 30      # PM2.5 baseline = median of 30 min before the event
PM_AFTER_MIN = 30     # look for the PM2.5 peak until 30 min after the event
RISE_UGM3 = 5.0       # event "shows in PM2.5" if peak - baseline >= this
PLOT_FILE = None      # e.g. "DataTable401" (None = file with the most events)
PLOT_DAYS = 3
# ====================================================================


def show_readme():
    """Print the README lines that explain the cooking sensor columns."""
    for f in DATA_DIR.glob("*readme*.txt"):
        print(f"--- {f.name}: lines about IBU / PTR / cooking ---")
        lines = f.read_text(errors="ignore").splitlines()
        for i, line in enumerate(lines):
            if re.search(r"IBU|PTR|cook|oven|burner|RHD", line, re.I):
                print(f"{i:4d}: {line.strip()}")
        print()


def load_file(path):
    df = pd.read_csv(path, low_memory=False)
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
    raw = df[TIME_COL].astype(str).str.strip()
    t = pd.to_datetime(raw, format=TIME_FORMAT, errors="coerce")
    if t.notna().mean() < 0.9:                      # other files use other formats
        t = pd.to_datetime(raw, format="mixed", errors="coerce")
    if t.notna().mean() < 0.9:
        print(f"  WARNING {path.name}: only {100 * t.notna().mean():.0f}% of times parsed; "
              f"examples: {raw.head(3).tolist()}")
    df["time"] = t
    df = df.dropna(subset=["time"]).sort_values("time").set_index("time")
    df = df[~df.index.duplicated()]
    for c in df.columns:
        if c != TIME_COL:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def cooking_on(df):
    """Return (on: bool Series, per-sensor ON Series dict)."""
    per = {}
    for c in [c for c in df.columns if c.startswith("IBU_")]:
        t = df[c]
        if t.notna().sum() == 0:
            continue
        base = t.rolling(f"{BASELINE_MIN}min", min_periods=10).quantile(0.10).shift(1)
        per[c] = (t - base) > DELTA_C
    for c in [c for c in df.columns if c.startswith("PTR_")]:
        if df[c].notna().sum():
            per[c] = df[c] > PTR_ON_W
    if not per:
        return pd.Series(False, index=df.index), per
    on = pd.concat(per, axis=1).fillna(False).any(axis=1)
    return on, per


def to_events(on, file_id):
    """Group consecutive ON minutes into events; drop short ones; merge close ones."""
    on = on.astype(bool)
    starts = on.index[on & ~on.shift(fill_value=False)]
    ends = on.index[on & ~on.shift(-1, fill_value=False)]
    ev = []
    for s, e in zip(starts, ends):
        if ev and s - ev[-1][1] <= pd.Timedelta(minutes=MIN_GAP_MIN):
            ev[-1][1] = e
        else:
            ev.append([s, e])
    ev = pd.DataFrame(ev, columns=["start", "end"])
    ev["start"] = pd.to_datetime(ev["start"])
    ev["end"] = pd.to_datetime(ev["end"])
    ev["duration_min"] = (ev.end - ev.start).dt.total_seconds() / 60 + 1
    ev = ev[ev.duration_min >= MIN_EVENT_MIN].reset_index(drop=True)
    ev.insert(0, "file", file_id)
    return ev


def pm_response(pm, ev):
    rows = []
    for _, r in ev.iterrows():
        b = pm[r.start - pd.Timedelta(minutes=PM_BASE_MIN): r.start].median()
        w = pm[r.start: r.end + pd.Timedelta(minutes=PM_AFTER_MIN)].dropna()
        peak = w.max() if len(w) else np.nan
        delay = (w.idxmax() - r.start).total_seconds() / 60 if len(w) else np.nan
        rows.append((b, peak, delay))
    ev = ev.copy()
    rows = pd.DataFrame(rows, index=ev.index, columns=["pm_baseline", "pm_peak", "peak_delay_min"])
    for c in rows.columns:
        ev[c] = rows[c].astype(float)
    ev["pm_rise"] = ev.pm_peak - ev.pm_baseline
    ev["shows_in_pm"] = ev.pm_rise >= RISE_UGM3
    return ev


def pick_pm(df):
    for c in PM_COLS:
        if c in df.columns and df[c].notna().mean() > 0.5:
            return c
    return None


def plot_file(df, ev, file_id, pm_col):
    t0 = ev.start.min().normalize() if len(ev) else df.index.min()
    t1 = t0 + pd.Timedelta(days=PLOT_DAYS)
    d = df[t0:t1]
    e = ev[(ev.start >= t0) & (ev.start < t1)]
    ibu = [c for c in d.columns if c.startswith("IBU_") and d[c].notna().any()]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    ax1.plot(d.index, d[pm_col], lw=0.8, color="#1f4e79")
    ax1.set_ylabel(f"{pm_col} (µg/m³)")
    for c in ibu:
        ax2.plot(d.index, d[c], lw=0.8, label=c)
    ax2.set_ylabel("Cooktop / oven temp (°C)")
    ax2.legend(loc="upper right", ncol=3, fontsize=8)
    for ax in (ax1, ax2):
        for _, r in e.iterrows():
            ax.axvspan(r.start, r.end, color="#e07b39", alpha=0.25)
    ax1.set_title(f"{file_id}: PM2.5 (top) and cooking sensors (bottom); "
                  f"orange = detected cooking")
    fig.tight_layout()
    path = OUT_DIR / f"{file_id}_pm25_cooking.png"
    fig.savefig(path, dpi=150)
    plt.show()
    print(f"Saved plot: {path}")


def run():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    show_readme()
    files = sorted(DATA_DIR.glob("DataTable*.csv"))
    print(f"{len(files)} home-week files\n")

    all_ev, summary, cache = [], [], {}
    for f in files:
        file_id = f.stem
        home = re.search(r"\d+", file_id).group()
        df = load_file(f)
        pm_col = pick_pm(df)
        on, per = cooking_on(df)
        ev = to_events(on, file_id)
        if pm_col:
            ev = pm_response(df[pm_col], ev)
        ev.insert(1, "home", home)
        all_ev.append(ev)
        days = (df.index.max() - df.index.min()).total_seconds() / 86400
        summary.append({
            "file": file_id, "home": home, "days": round(days, 1),
            "cook_sensors": ",".join(per.keys()) or "NONE",
            "pm_col": pm_col or "NONE",
            "n_events": len(ev),
            "events_per_day": round(len(ev) / days, 1) if days else np.nan,
            "pct_minutes_cooking": round(100 * on.mean(), 2),
            "pct_events_pm_rise": round(100 * ev.shows_in_pm.mean(), 0) if len(ev) and pm_col else np.nan,
            "median_pm_rise": round(ev.pm_rise.median(), 1) if len(ev) and pm_col else np.nan,
            "median_peak_delay_min": round(ev.peak_delay_min.median(), 0) if len(ev) and pm_col else np.nan,
        })
        cache[file_id] = (df, ev, pm_col)

    ev = pd.concat(all_ev, ignore_index=True)
    s = pd.DataFrame(summary)
    ev.to_csv(OUT_DIR / "cooking_events.csv", index=False)
    s.to_csv(OUT_DIR / "summary_per_file.csv", index=False)

    pd.set_option("display.width", 200)
    print(s.to_string(index=False))

    print("\n================ SUMMARY ================")
    print(f"Home-week files:                  {len(s)}  ({s.home.nunique()} homes)")
    print(f"Files with no cooking sensors:    {(s.cook_sensors == 'NONE').sum()}")
    print(f"Files with >= 10 cooking events:  {(s.n_events >= 10).sum()}")
    print(f"Total cooking events:             {len(ev)}")
    print(f"Median events per day:           {s.events_per_day.median()}")
    print(f"Cooking share of minutes:         {s.pct_minutes_cooking.median()}% (median per file)")
    if "shows_in_pm" in ev:
        print(f"Events with PM2.5 rise >= {RISE_UGM3}:   {100 * ev.shows_in_pm.mean():.0f}%")
        print(f"Median delay to PM2.5 peak:       {ev.peak_delay_min.median():.0f} min")
    print("\nGo/no-go guide:")
    print("  GO    most files have >= 10 events and 1-4 events/day, and many events show a PM2.5 rise")
    print("  TUNE  far too many events/day -> raise DELTA_C; almost none -> lower DELTA_C")
    print("  NO-GO most files have no cooking sensors or no events -> fallback (occupancy main task)")

    pick = PLOT_FILE or s.sort_values("n_events", ascending=False).file.iloc[0]
    df, e, pm_col = cache[pick]
    if pm_col:
        plot_file(df, e, pick, pm_col)
    return s, ev


summary, events = run()

In [ ]:
"""
D7 (BAVIAQ) cooking labels, version 2  -  MECE 694 project, Week 1
Colab: paste this whole file into one cell and run it.

Changes from version 1
  1. Events end when the burner/oven starts COOLING (temperature falling),
     not when it finally drops back near room temperature.
  2. Broken sensors are flagged and left out:
       - "stuck hot": above the threshold for more than MAX_ON_SHARE of the time
       - "flat":      almost no variation (std below FLAT_STD_C)
  3. Saves a per-minute label file for every home-week (used later in stages 3-4).
  4. Saves plots for several home-weeks, including the suspicious ones.

Outputs (in your Drive, under Results/MECE694_labels_v2/)
  labels/DataTableXXX_labels.csv.gz   per-minute: time, file, home, cooking (0/1)
  cooking_events.csv                  one row per event
  summary_per_file.csv                one row per home-week
  sensor_flags.csv                    sensors left out and why
  plots/*.png                         PM2.5 + cooking sensors with detected events
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# ============================== CONFIG ==============================
BASE = Path("/content/drive/MyDrive/IAQ Research/BAVIAQ_Residential_Homes")
DATA_DIR = BASE / "Raw_Data/Extracted/IAQ_Activity_Monitoring/IAQ_Activity_Monitoring"
OUT_DIR = BASE / "Results/MECE694_labels_v2"

TIME_COL = "Time"
TIME_FORMAT = "%m/%d/%Y %H:%M"
PM_COLS = ["PMin", "AVP_IN1_PM25"]      # first one with >50% data is used

# --- Labelling ---
DELTA_C = 8.0          # sensor must be this many deg C above its baseline to count
BASELINE_MIN = 180     # baseline = 10th percentile of the previous 180 min
SLOPE_MIN = 5          # slope measured over 5 min
COOL_RATE = -0.5       # deg C per min; falling faster than this = cooling, not cooking
RISE_RATE = 0.3        # deg C per min; an event must heat up at least this fast at some point
PTR_ON_W = 20.0        # power loggers (PTR_) count as cooking above 20 W
MIN_EVENT_MIN = 3      # drop events shorter than 3 min
MIN_GAP_MIN = 10       # merge events less than 10 min apart

# --- Broken-sensor checks ---
MAX_ON_SHARE = 0.30    # a sensor "on" more than 30% of the week is treated as stuck
FLAT_STD_C = 0.3       # a sensor with std below 0.3 deg C is treated as flat

# --- PM2.5 response check ---
PM_BASE_MIN = 30
PM_AFTER_MIN = 30
RISE_UGM3 = 5.0

# --- Plots ---
PLOT_FILES = ["DataTable435_OFF", "DataTable447_OFF", "DataTable438_ON",
              "DataTable452", "DataTable406"]   # home-weeks to plot
PLOT_DAYS = 3
SHOW_PLOTS = True      # False = only save plots to Drive
# ====================================================================


def load_file(path):
    df = pd.read_csv(path, low_memory=False)
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
    raw = df[TIME_COL].astype(str).str.strip()
    t = pd.to_datetime(raw, format=TIME_FORMAT, errors="coerce")
    if t.notna().mean() < 0.9:
        t = pd.to_datetime(raw, format="mixed", errors="coerce")
    if t.notna().mean() < 0.9:
        print(f"  WARNING {path.name}: only {100 * t.notna().mean():.0f}% of times parsed")
    df["time"] = t
    df = df.dropna(subset=["time"]).sort_values("time").set_index("time")
    df = df[~df.index.duplicated()]
    for c in df.columns:
        if c != TIME_COL:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def sensor_on(temp):
    """ON = well above baseline AND not cooling down."""
    base = temp.rolling(f"{BASELINE_MIN}min", min_periods=10).quantile(0.10).shift(1)
    above = (temp - base) > DELTA_C
    slope = temp.diff(SLOPE_MIN) / SLOPE_MIN            # deg C per min
    cooling = slope < COOL_RATE
    on = (above & ~cooling).fillna(False)
    # keep only ON stretches where the temperature actually rises (real heating),
    # which drops the slow tail of a cool-down
    seg = (on != on.shift()).cumsum()
    max_slope = slope.where(on).groupby(seg).transform("max")
    return on & (max_slope > RISE_RATE)


def cooking_on(df, file_id):
    per, flags = {}, []
    for c in [c for c in df.columns if c.startswith("IBU_")]:
        t = df[c]
        if t.notna().sum() < 60:
            continue
        if t.std() < FLAT_STD_C:
            flags.append((file_id, c, "flat", round(t.std(), 2)))
            continue
        hot_share = (t > t.quantile(0.05) + DELTA_C).mean()   # hot vs. its own coolest level
        if hot_share > MAX_ON_SHARE:
            flags.append((file_id, c, "stuck hot", round(hot_share, 2)))
            continue
        on = sensor_on(t)
        if on.mean() > MAX_ON_SHARE:
            flags.append((file_id, c, "stuck hot", round(on.mean(), 2)))
            continue
        per[c] = on
    for c in [c for c in df.columns if c.startswith("PTR_")]:
        if df[c].notna().sum():
            on = (df[c] > PTR_ON_W).fillna(False)
            if on.mean() > MAX_ON_SHARE:
                flags.append((file_id, c, "stuck on", round(on.mean(), 2)))
                continue
            per[c] = on
    if not per:
        return pd.Series(False, index=df.index), per, flags
    on = pd.concat(per, axis=1).fillna(False).any(axis=1)
    return on, per, flags


def to_events(on, file_id):
    on = on.astype(bool)
    starts = on.index[on & ~on.shift(fill_value=False)]
    ends = on.index[on & ~on.shift(-1, fill_value=False)]
    ev = []
    for s, e in zip(starts, ends):
        if ev and s - ev[-1][1] <= pd.Timedelta(minutes=MIN_GAP_MIN):
            ev[-1][1] = e
        else:
            ev.append([s, e])
    ev = pd.DataFrame(ev, columns=["start", "end"])
    ev["start"] = pd.to_datetime(ev["start"])
    ev["end"] = pd.to_datetime(ev["end"])
    ev["duration_min"] = (ev.end - ev.start).dt.total_seconds() / 60 + 1
    ev = ev[ev.duration_min >= MIN_EVENT_MIN].reset_index(drop=True)
    ev.insert(0, "file", file_id)
    return ev


def events_to_minutes(index, ev):
    """Per-minute 0/1 label from the final (merged, filtered) events."""
    lab = pd.Series(0, index=index, dtype="int8")
    for _, r in ev.iterrows():
        lab[r.start:r.end] = 1
    return lab


def pm_response(pm, ev):
    rows = []
    for _, r in ev.iterrows():
        b = pm[r.start - pd.Timedelta(minutes=PM_BASE_MIN): r.start].median()
        w = pm[r.start: r.end + pd.Timedelta(minutes=PM_AFTER_MIN)].dropna()
        peak = w.max() if len(w) else np.nan
        delay = (w.idxmax() - r.start).total_seconds() / 60 if len(w) else np.nan
        rows.append((b, peak, delay))
    ev = ev.copy()
    rows = pd.DataFrame(rows, index=ev.index, columns=["pm_baseline", "pm_peak", "peak_delay_min"])
    for c in rows.columns:
        ev[c] = rows[c].astype(float)
    ev["pm_rise"] = ev.pm_peak - ev.pm_baseline
    ev["shows_in_pm"] = ev.pm_rise >= RISE_UGM3
    return ev


def pick_pm(df):
    for c in PM_COLS:
        if c in df.columns and df[c].notna().mean() > 0.5:
            return c
    return None


def plot_file(df, ev, file_id, pm_col, flagged):
    t0 = ev.start.min().normalize() if len(ev) else df.index.min().normalize()
    t1 = t0 + pd.Timedelta(days=PLOT_DAYS)
    d = df[t0:t1]
    e = ev[(ev.start >= t0) & (ev.start < t1)]
    ibu = [c for c in d.columns if c.startswith("IBU_") and d[c].notna().any()]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    if pm_col:
        ax1.plot(d.index, d[pm_col], lw=0.8, color="#1f4e79")
        ax1.set_ylabel(f"{pm_col} (µg/m³)")
    for c in ibu:
        style = ":" if c in flagged else "-"
        ax2.plot(d.index, d[c], lw=0.8, ls=style,
                 label=f"{c} (left out)" if c in flagged else c)
    ax2.set_ylabel("Cooktop / oven temp (°C)")
    ax2.legend(loc="upper right", ncol=3, fontsize=8)
    for ax in (ax1, ax2):
        for _, r in e.iterrows():
            ax.axvspan(r.start, r.end, color="#e07b39", alpha=0.25)
    ax1.set_title(f"{file_id}: PM2.5 (top), cooking sensors (bottom); "
                  f"orange = detected cooking; dotted = sensor left out")
    fig.tight_layout()
    path = OUT_DIR / "plots" / f"{file_id}_pm25_cooking.png"
    fig.savefig(path, dpi=150)
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)
    print(f"Saved plot: {path}")


def run():
    for sub in ["", "labels", "plots"]:
        (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
    files = sorted(DATA_DIR.glob("DataTable*.csv"))
    print(f"{len(files)} home-week files\n")

    all_ev, summary, all_flags = [], [], []
    for f in files:
        file_id = f.stem
        home = re.search(r"\d+", file_id).group()
        df = load_file(f)
        pm_col = pick_pm(df)
        on, per, flags = cooking_on(df, file_id)
        all_flags += flags
        ev = to_events(on, file_id)
        if pm_col:
            ev = pm_response(df[pm_col], ev)
        ev.insert(1, "home", home)
        all_ev.append(ev)

        label = events_to_minutes(df.index, ev)
        pd.DataFrame({"time": df.index, "file": file_id, "home": home,
                      "cooking": label.values}).to_csv(
            OUT_DIR / "labels" / f"{file_id}_labels.csv.gz", index=False)

        days = (df.index.max() - df.index.min()).total_seconds() / 86400
        has = len(ev) > 0 and pm_col is not None
        summary.append({
            "file": file_id, "home": home, "days": round(days, 1),
            "sensors_used": ",".join(per.keys()) or "NONE",
            "sensors_left_out": ",".join(c for _, c, _, _ in flags) or "",
            "pm_col": pm_col or "NONE",
            "n_events": len(ev),
            "events_per_day": round(len(ev) / days, 1) if days else np.nan,
            "median_event_min": round(ev.duration_min.median(), 0) if len(ev) else np.nan,
            "pct_minutes_cooking": round(100 * label.mean(), 2),
            "pct_events_pm_rise": round(100 * ev.shows_in_pm.mean(), 0) if has else np.nan,
            "median_pm_rise": round(ev.pm_rise.median(), 1) if has else np.nan,
        })
        if file_id in PLOT_FILES:
            plot_file(df, ev, file_id, pm_col, {c for _, c, _, _ in flags})

    ev = pd.concat(all_ev, ignore_index=True)
    s = pd.DataFrame(summary)
    fl = pd.DataFrame(all_flags, columns=["file", "sensor", "problem", "value"])
    ev.to_csv(OUT_DIR / "cooking_events.csv", index=False)
    s.to_csv(OUT_DIR / "summary_per_file.csv", index=False)
    fl.to_csv(OUT_DIR / "sensor_flags.csv", index=False)

    pd.set_option("display.width", 220)
    print(s.drop(columns=["sensors_used"]).to_string(index=False))
    print("\nSensors left out:")
    print(fl.to_string(index=False) if len(fl) else "  none")

    print("\n================ SUMMARY ================")
    print(f"Home-week files:                  {len(s)}  ({s.home.nunique()} homes)")
    print(f"Files with no usable sensors:     {(s.sensors_used == 'NONE').sum()}")
    print(f"Files with >= 10 cooking events:  {(s.n_events >= 10).sum()}")
    print(f"Total cooking events:             {len(ev)}")
    print(f"Median events per day:            {s.events_per_day.median()}")
    print(f"Median event length:              {ev.duration_min.median():.0f} min")
    print(f"Cooking share of minutes:         {s.pct_minutes_cooking.median()}% (median per file)")
    if "shows_in_pm" in ev:
        print(f"Events with PM2.5 rise >= {RISE_UGM3}:   {100 * ev.shows_in_pm.mean():.0f}%")
    print(f"\nAll outputs saved in: {OUT_DIR}")
    return s, ev, fl


summary, events, flags = run()